# Compare + Tuning + MLflow

This notebook compares baselines, runs targeted tuning (LGBM/XGB), logs MLflow, and writes reports/best_model.json.


## 4.1 Config
Set paths, seeds, and tuning budget.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / 'notebooks' / 'notebook_params.yaml'
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Config not found: {CONFIG_PATH}')

try:
    import yaml
except ImportError as exc:
    raise ImportError('pyyaml is required to load notebook_params.yaml') from exc

config = yaml.safe_load(CONFIG_PATH.read_text()) or {}
cfg = config.get('compare_tuning', {})

DATA_PATH = PROJECT_ROOT / cfg.get('DATA_PATH', 'data/data_final.parquet')
ARTIFACTS_PATH = PROJECT_ROOT / cfg.get('ARTIFACTS_PATH', 'artifacts/preprocessor.joblib')
REPORT_PATH = PROJECT_ROOT / cfg.get('REPORT_PATH', 'reports/best_model.json')

SEED = int(cfg.get('SEED', 42))
SAMPLE_SIZE = int(cfg.get('SAMPLE_SIZE', 50000))  # 0 for full dataset
TEST_SIZE = float(cfg.get('TEST_SIZE', 0.2))
CV_SPLITS = int(cfg.get('CV_SPLITS', 5))
N_JOBS = int(cfg.get('N_JOBS', 1))
TUNING_TRIALS = int(cfg.get('TUNING_TRIALS', 30))
LOG_ARTIFACTS = bool(cfg.get('LOG_ARTIFACTS', True))
EXPERIMENT_NAME = str(cfg.get('EXPERIMENT_NAME', 'Model-Compare-Tuning'))
SELECTION_RULE = str(cfg.get('SELECTION_RULE', '0.7*f1_best + 0.3*pr_auc'))

if not DATA_PATH.exists():
    raise FileNotFoundError(f'{DATA_PATH} not found. Run the exploration notebook.')


## 4.2 Imports


In [2]:
import json
import re
from dataclasses import dataclass
from typing import Any

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

from app.main import (
    DAYS_EMPLOYED_SENTINEL,
    ENGINEERED_SOURCES,
    IGNORE_FEATURES,
    MISSING_INDICATOR_MIN_RATE,
    OUTLIER_COLUMNS,
    OUTLIER_LOWER_Q,
    OUTLIER_UPPER_Q,
    _apply_correlated_imputation,
    _validate_numeric_inputs,
    add_missingness_indicators,
    apply_outlier_clipping,
    compute_outlier_bounds,
    load_preprocessor,
    new_features_creation,
    select_missing_indicator_columns,
)

from notebooks.dev_preprocess_utils import preprocess_for_training


## Helpers


In [3]:
@dataclass
class ModelSpec:
    name: str
    model: Any
    needs_sanitized_features: bool = False


@dataclass
class TrialResult:
    model_name: str
    params: dict[str, Any]
    best_threshold: float
    f1_best: float
    pr_auc: float
    roc_auc: float
    score_composite: float


def sanitize_feature_names(columns: list[str]) -> list[str]:
    cleaned: list[str] = []
    seen: dict[str, int] = {}
    for col in columns:
        base = re.sub(r'[^0-9a-zA-Z_]+', '_', str(col)).strip('_')
        if not base:
            base = 'feature'
        if base[0].isdigit():
            base = f'f_{base}'
        if base in seen:
            seen[base] += 1
            base = f'{base}_{seen[base]}'
        else:
            seen[base] = 0
        cleaned.append(base)
    return cleaned


def best_threshold_for_f1(y_true: np.ndarray, y_proba: np.ndarray) -> tuple[float, float, float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    if thresholds.size == 0:
        preds = (y_proba >= 0.5).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        prec = precision_score(y_true, preds, zero_division=0)
        rec = recall_score(y_true, preds, zero_division=0)
        return 0.5, float(f1), float(prec), float(rec)
    f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])
    f1_scores = np.nan_to_num(f1_scores, nan=0.0)
    best_idx = int(np.argmax(f1_scores))
    return (
        float(thresholds[best_idx]),
        float(f1_scores[best_idx]),
        float(precision[best_idx]),
        float(recall[best_idx]),
    )


def composite_score(f1_best: float, pr_auc: float, weight_f1: float = 0.7) -> float:
    return float(weight_f1 * f1_best + (1.0 - weight_f1) * pr_auc)


def evaluate_model(
    spec: ModelSpec,
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    cv: StratifiedKFold,
    n_jobs: int,
) -> tuple[dict[str, float], dict[str, Any]]:
    model = spec.model
    if spec.needs_sanitized_features:
        sanitized = sanitize_feature_names(list(X_train.columns))
        X_train = X_train.copy()
        X_test = X_test.copy()
        X_train.columns = sanitized
        X_test.columns = sanitized

    oof_proba = cross_val_predict(
        model,
        X_train,
        y_train,
        cv=cv,
        method='predict_proba',
        n_jobs=n_jobs,
    )[:, 1]
    best_threshold, f1_best, precision_best, recall_best = best_threshold_for_f1(y_train, oof_proba)
    pr_auc = float(average_precision_score(y_train, oof_proba))
    roc_auc = float(roc_auc_score(y_train, oof_proba))
    score = composite_score(f1_best, pr_auc)

    model.fit(X_train, y_train)
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= best_threshold).astype(int)

    precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, test_proba)
    tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()

    test_metrics = {
        'test_f1': float(f1_score(y_test, test_pred, zero_division=0)),
        'test_precision': float(precision_score(y_test, test_pred, zero_division=0)),
        'test_recall': float(recall_score(y_test, test_pred, zero_division=0)),
        'test_pr_auc': float(average_precision_score(y_test, test_proba)),
        'test_roc_auc': float(roc_auc_score(y_test, test_proba)),
    }

    artifacts = {
        'confusion_matrix': {
            'tn': int(tn),
            'fp': int(fp),
            'fn': int(fn),
            'tp': int(tp),
        },
        'pr_curve': {
            'precision': precision_curve.tolist(),
            'recall': recall_curve.tolist(),
            'thresholds': thresholds.tolist(),
        },
    }

    return (
        {
            'best_threshold': float(best_threshold),
            'f1_best': float(f1_best),
            'precision_best': float(precision_best),
            'recall_best': float(recall_best),
            'pr_auc': float(pr_auc),
            'roc_auc': float(roc_auc),
            'score_composite': float(score),
            **test_metrics,
        },
        artifacts,
    )


## 4.2 Load data and split


In [4]:
df = pd.read_parquet(DATA_PATH)
df = df[df['TARGET'].notna()].copy()
df['TARGET'] = df['TARGET'].astype(int)

if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
    df, _ = train_test_split(
        df,
        train_size=SAMPLE_SIZE,
        stratify=df['TARGET'],
        random_state=SEED,
    )

preprocessor = load_preprocessor(DATA_PATH, ARTIFACTS_PATH)
X_all = preprocess_for_training(df, preprocessor)
y_all = df['TARGET'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=SEED,
)

pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight = float(neg / max(pos, 1))

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)

print('train size:', len(y_train), 'test size:', len(y_test))
print('scale_pos_weight:', scale_pos_weight)


train size: 40000 test size: 10000
scale_pos_weight: 11.387736141220191


## 4.3 Baselines (compare)


In [5]:
specs = [
    ModelSpec(
        name='HistGB',
        model=HistGradientBoostingClassifier(
            max_depth=4,
            max_iter=200,
            learning_rate=0.05,
            min_samples_leaf=30,
            l2_regularization=0.0,
            class_weight='balanced',
            random_state=SEED,
        ),
    ),
]

if LGBMClassifier is not None:
    specs.append(
        ModelSpec(
            name='LightGBM',
            model=LGBMClassifier(
                objective='binary',
                n_estimators=400,
                learning_rate=0.05,
                num_leaves=64,
                min_child_samples=100,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                scale_pos_weight=scale_pos_weight,
                random_state=SEED,
                n_jobs=N_JOBS,
                verbosity=-1,
            ),
            needs_sanitized_features=True,
        )
    )
else:
    print('LightGBM not installed; skipping LGBM baseline.')

if XGBClassifier is not None:
    specs.append(
        ModelSpec(
            name='XGBoost',
            model=XGBClassifier(
                objective='binary:logistic',
                eval_metric='logloss',
                n_estimators=400,
                learning_rate=0.05,
                max_depth=5,
                min_child_weight=10,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                scale_pos_weight=scale_pos_weight,
                random_state=SEED,
                n_jobs=N_JOBS,
                tree_method='hist',
                verbosity=0,
            ),
            needs_sanitized_features=True,
        )
    )
else:
    print('XGBoost not installed; skipping XGB baseline.')

mlflow.set_experiment(EXPERIMENT_NAME)
baseline_results = []

for spec in specs:
    run_name = f'baseline_{spec.name}'
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param('model_name', spec.name)
        mlflow.log_param('cv_splits', CV_SPLITS)
        mlflow.log_param('sample_size', SAMPLE_SIZE)
        mlflow.log_param('random_state', SEED)
        mlflow.log_params(spec.model.get_params())

        metrics, artifacts = evaluate_model(
            spec,
            X_train,
            y_train,
            X_test,
            y_test,
            cv=cv,
            n_jobs=N_JOBS,
        )
        mlflow.log_metrics(metrics)

        if LOG_ARTIFACTS:
            mlflow.log_dict(
                artifacts['confusion_matrix'],
                f'artifacts/{spec.name}_confusion_matrix.json',
            )
            mlflow.log_dict(
                artifacts['pr_curve'],
                f'artifacts/{spec.name}_pr_curve.json',
            )

        baseline_results.append({
            'model_name': spec.name.lower(),
            'params': spec.model.get_params(),
            'metrics': metrics,
        })
        print(spec.name, metrics['score_composite'])


/Users/steph/Code/Python/Jupyter/OCR_projet06/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)


HistGB 0.26528725570364303


LightGBM 0.2589803807514172


XGBoost 0.26498399243689497


## 4.4 Tuning (LGBM + XGB)


In [6]:
def run_tuning(
    *,
    model_name: str,
    make_model,
    sample_params,
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    trials: int,
    cv: StratifiedKFold,
    n_jobs: int,
    base_spw: float,
    rng: np.random.Generator,
) -> TrialResult:
    def _make_model(params_in: dict[str, Any]):
        params_out = dict(params_in)
        if 'n_jobs' not in params_out:
            params_out['n_jobs'] = n_jobs
        try:
            return make_model(params_out), params_out
        except TypeError:
            if 'n_jobs' in params_out:
                params_out.pop('n_jobs', None)
                return make_model(params_out), params_out
            raise

    best: TrialResult | None = None
    with mlflow.start_run(run_name=f'tune_{model_name}'):
        mlflow.log_param('model', model_name)
        mlflow.log_param('trials', trials)
        mlflow.log_param('cv_splits', CV_SPLITS)
        mlflow.log_param('scale_pos_weight_base', base_spw)
        for idx in range(trials):
            params = sample_params(rng, base_spw)
            model, used_params = _make_model(params)
            with mlflow.start_run(run_name=f'{model_name}_trial_{idx + 1:03d}', nested=True):
                mlflow.log_params(used_params)
                oof_proba = cross_val_predict(
                    model,
                    X_train,
                    y_train,
                    cv=cv,
                    method='predict_proba',
                    n_jobs=n_jobs,
                )[:, 1]
                best_threshold, f1_best, precision_best, recall_best = best_threshold_for_f1(y_train, oof_proba)
                pr_auc = float(average_precision_score(y_train, oof_proba))
                roc_auc = float(roc_auc_score(y_train, oof_proba))
                score = composite_score(f1_best, pr_auc)

                mlflow.log_metric('best_threshold', best_threshold)
                mlflow.log_metric('f1_best', f1_best)
                mlflow.log_metric('precision_best', precision_best)
                mlflow.log_metric('recall_best', recall_best)
                mlflow.log_metric('pr_auc', pr_auc)
                mlflow.log_metric('roc_auc', roc_auc)
                mlflow.log_metric('score_composite', score)

                if best is None or score > best.score_composite:
                    best = TrialResult(
                        model_name=model_name,
                        params=used_params,
                        best_threshold=float(best_threshold),
                        f1_best=float(f1_best),
                        pr_auc=float(pr_auc),
                        roc_auc=float(roc_auc),
                        score_composite=float(score),
                    )
        if best is None:
            raise RuntimeError(f'No trials executed for {model_name}')

        best_model, _ = _make_model(best.params)
        best_model.fit(X_train, y_train)
        test_proba = best_model.predict_proba(X_test)[:, 1]
        test_pred = (test_proba >= best.best_threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()

        mlflow.log_metric('test_f1', float(f1_score(y_test, test_pred, zero_division=0)))
        mlflow.log_metric('test_precision', float(precision_score(y_test, test_pred, zero_division=0)))
        mlflow.log_metric('test_recall', float(recall_score(y_test, test_pred, zero_division=0)))
        mlflow.log_metric('test_pr_auc', float(average_precision_score(y_test, test_proba)))
        mlflow.log_metric('test_roc_auc', float(roc_auc_score(y_test, test_proba)))
        mlflow.log_metric('test_cm_tn', float(tn))
        mlflow.log_metric('test_cm_fp', float(fp))
        mlflow.log_metric('test_cm_fn', float(fn))
        mlflow.log_metric('test_cm_tp', float(tp))

    return best


In [7]:
tuning_results = []
rng = np.random.default_rng(SEED)

histgb_trials = max(10, min(30, TUNING_TRIALS))
tuning_results.append(
    run_tuning(
        model_name='HistGB',
        make_model=lambda params: HistGradientBoostingClassifier(**params),
        sample_params=lambda rng, base_spw: {
            'max_depth': int(rng.integers(2, 6)),
            'max_iter': int(rng.integers(150, 401)),
            'learning_rate': float(10 ** rng.uniform(np.log10(0.01), np.log10(0.1))),
            'min_samples_leaf': int(rng.integers(20, 101)),
            'l2_regularization': float(rng.uniform(0.0, 1.0)),
            'class_weight': 'balanced',
            'random_state': SEED,
        },
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        trials=histgb_trials,
        cv=cv,
        n_jobs=N_JOBS,
        base_spw=scale_pos_weight,
        rng=rng,
    )
)

if LGBMClassifier is None:
    print('LightGBM not installed; skipping tuning.')
else:
    X_train_use = X_train.copy()
    X_test_use = X_test.copy()
    sanitized = sanitize_feature_names(list(X_train_use.columns))
    X_train_use.columns = sanitized
    X_test_use.columns = sanitized
    tuning_results.append(
        run_tuning(
            model_name='LightGBM',
            make_model=lambda params: LGBMClassifier(**params),
            sample_params=lambda rng, base_spw: {
                'objective': 'binary',
                'n_estimators': int(rng.integers(200, 801)),
                'learning_rate': float(10 ** rng.uniform(np.log10(0.005), np.log10(0.2))),
                'num_leaves': int(rng.integers(32, 257)),
                'min_child_samples': int(rng.integers(50, 301)),
                'subsample': float(rng.uniform(0.6, 1.0)),
                'colsample_bytree': float(rng.uniform(0.6, 1.0)),
                'reg_alpha': float(rng.uniform(0.0, 5.0)),
                'reg_lambda': float(rng.uniform(0.5, 20.0)),
                'scale_pos_weight': float(base_spw * rng.uniform(0.7, 1.3)),
                'verbosity': -1,
                'random_state': SEED,
            },
            X_train=X_train_use,
            y_train=y_train,
            X_test=X_test_use,
            y_test=y_test,
            trials=TUNING_TRIALS,
            cv=cv,
            n_jobs=N_JOBS,
            base_spw=scale_pos_weight,
            rng=rng,
        )
    )

if XGBClassifier is None:
    print('XGBoost not installed; skipping tuning.')
else:
    X_train_use = X_train.copy()
    X_test_use = X_test.copy()
    sanitized = sanitize_feature_names(list(X_train_use.columns))
    X_train_use.columns = sanitized
    X_test_use.columns = sanitized
    tuning_results.append(
        run_tuning(
            model_name='XGBoost',
            make_model=lambda params: XGBClassifier(**params),
            sample_params=lambda rng, base_spw: {
                'objective': 'binary:logistic',
                'eval_metric': 'logloss',
                'n_estimators': int(rng.integers(200, 801)),
                'learning_rate': float(10 ** rng.uniform(np.log10(0.01), np.log10(0.2))),
                'max_depth': int(rng.integers(3, 9)),
                'min_child_weight': float(rng.integers(5, 31)),
                'subsample': float(rng.uniform(0.6, 1.0)),
                'colsample_bytree': float(rng.uniform(0.6, 1.0)),
                'gamma': float(rng.uniform(0.0, 5.0)),
                'reg_alpha': float(rng.uniform(0.0, 5.0)),
                'reg_lambda': float(rng.uniform(0.5, 20.0)),
                'scale_pos_weight': float(base_spw * rng.uniform(0.7, 1.3)),
                'random_state': SEED,
                'tree_method': 'hist',
                'verbosity': 0,
            },
            X_train=X_train_use,
            y_train=y_train,
            X_test=X_test_use,
            y_test=y_test,
            trials=TUNING_TRIALS,
            cv=cv,
            n_jobs=N_JOBS,
            base_spw=scale_pos_weight,
            rng=rng,
        )
    )


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


...

/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


/var/folders/by/r6dmty813rxgqdnr7hvtrl480000gn/T/ipykernel_90063/925434510.py:45: RuntimeWarning: invalid value encountered in divide
  f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1])


## 4.5 Selection and export


In [8]:
candidates = []
for item in baseline_results:
    metrics = item['metrics']
    candidates.append({
        'model_name': item['model_name'],
        'params': item['params'],
        'threshold': metrics['best_threshold'],
        'metrics': {
            'f1_best': metrics['f1_best'],
            'pr_auc': metrics['pr_auc'],
            'roc_auc': metrics['roc_auc'],
        },
        'score_composite': metrics['score_composite'],
    })

for item in tuning_results:
    candidates.append({
        'model_name': item.model_name.lower(),
        'params': item.params,
        'threshold': item.best_threshold,
        'metrics': {
            'f1_best': item.f1_best,
            'pr_auc': item.pr_auc,
            'roc_auc': item.roc_auc,
        },
        'score_composite': item.score_composite,
    })

if not candidates:
    raise RuntimeError('No candidates available for selection.')

champion = max(candidates, key=lambda c: c['score_composite'])

stat = DATA_PATH.stat()
best_model = {
    'model_name': champion['model_name'],
    'params': champion['params'],
    'threshold': float(champion['threshold']),
    'metrics': champion['metrics'],
    'selection_rule': SELECTION_RULE,
    'data_fingerprint': {
        'path': str(DATA_PATH),
        'mtime': stat.st_mtime,
        'size_bytes': stat.st_size,
    },
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(best_model, indent=2), encoding='utf-8')

print('Selected:', champion['model_name'])
print('Score:', champion['score_composite'])
print(f'Wrote {REPORT_PATH}')


Selected: xgboost
Score: 0.2770859284954027
Wrote /Users/steph/Code/Python/Jupyter/OCR_projet06/reports/best_model.json
